# Spot Tolerance Cost Analysis — Scenario B

In [ ]:
import json
import os
import re
import glob
from collections import defaultdict
from itertools import groupby

import pandas as pd

# ── Configuration ──────────────────────────────────────────────────────────────
SCENARIO = "B"
MODEL_KEY = "llama3-70b"          # e.g., "llama3-70b", "qwen3-32b"
BENCHMARK_DURATION_MIN = 60       # Adjustable: Qwen may finish in < 60 min

# Derived paths
_model_key_underscore = MODEL_KEY.replace("-", "_")
PRICES_FILE   = f"results/prices_scenario_{SCENARIO}.json"
EVENTS_FILE   = f"spot_trace_events_scenario_{SCENARIO}.json"
NODES_FILE    = f"nodes_scenario_{SCENARIO}.json"
PIPELINE_FILE = f"{MODEL_KEY}/pipelines_{_model_key_underscore}_scenario_{SCENARIO}.json"
LOG_DIR       = f"{MODEL_KEY}/offline/scenario_{SCENARIO}/logs"
TRACE_DIR     = f"results/{MODEL_KEY}/offline/scenario_{SCENARIO}/Trace"

# AZ selection per instance type.
# Set a specific AZ to use that AZ's spot price.
# Leave empty {} to compute avg/min/max across all AZs.
AZ_SELECTION = {
    # "g5.12xlarge": "us-west-2a",
    # "g6.12xlarge": "us-west-2a",
    # "g6e.xlarge": "us-west-2a",
}

# Approaches
APPROACHES = ["only_ondemand", "shuntserve", "concurrent_initialization",
              "request_migration", "no_handle"]
OVERLAP_APPROACHES = {"shuntserve", "concurrent_initialization"}

print(f"Scenario: {SCENARIO}, Model: {MODEL_KEY}, Duration: {BENCHMARK_DURATION_MIN} min")
print(f"Pipeline file: {PIPELINE_FILE}")
print(f"Trace dir: {TRACE_DIR}")

In [ ]:
# ── Load & Parse Data ──────────────────────────────────────────────────────────

with open(PRICES_FILE) as f:
    prices_data = json.load(f)
with open(EVENTS_FILE) as f:
    events_raw = json.load(f)["events"]
with open(NODES_FILE) as f:
    nodes_data = json.load(f)
with open(PIPELINE_FILE) as f:
    pipeline_data = json.load(f)

# ── Helper: node name → group key and instance type ───────────────────────────

def node_to_group(name: str) -> str:
    """spot_g6e_xlarge_node_ip_1 → spot_g6e_xlarge"""
    return re.sub(r"_node_ip_\d+$", "", name)

def group_to_instance_type(group: str) -> str:
    """spot_g6e_xlarge → g6e.xlarge"""
    bare = re.sub(r"^(spot|on_demand)_", "", group)
    return re.sub(r"_(\d*xlarge)$", r".\1", bare)

def is_spot_group(group: str) -> bool:
    return group.startswith("spot_")

# ── Parse node groups from nodes.json ─────────────────────────────────────────

all_node_groups = defaultdict(int)  # group_key → count
for name in nodes_data:
    all_node_groups[node_to_group(name)] += 1

print("All node groups:")
for g, c in sorted(all_node_groups.items()):
    print(f"  {g}: {c}  ({group_to_instance_type(g)})")

# ── Parse initial active nodes from pipeline.json ─────────────────────────────

initial_active = defaultdict(int)  # group_key → count
for pipeline in pipeline_data["pipelines"]:
    for node_name, _layers in pipeline["node_layer_mapping"]:
        initial_active[node_to_group(node_name)] += 1

print("\nInitial active (from pipeline):")
for g, c in sorted(initial_active.items()):
    print(f"  {g}: {c}")

# ── Per instance type: initial (spot_active, od_active) ───────────────────────

instance_types = set()
for g in all_node_groups:
    instance_types.add(group_to_instance_type(g))

initial_state = {}  # instance_type → {"spot_active": N, "od_active": N, "spot_total": N, "od_total": N}
for itype in sorted(instance_types):
    spot_total = od_total = spot_active = od_active = 0
    for g, cnt in all_node_groups.items():
        if group_to_instance_type(g) == itype:
            if is_spot_group(g):
                spot_total += cnt
                spot_active += initial_active.get(g, 0)
            else:
                od_total += cnt
                od_active += initial_active.get(g, 0)
    initial_state[itype] = {
        "spot_active": spot_active, "od_active": od_active,
        "spot_total": spot_total, "od_total": od_total,
    }

print("\nInitial state per instance type:")
for itype, s in initial_state.items():
    print(f"  {itype}: spot={s['spot_active']}/{s['spot_total']}, od={s['od_active']}/{s['od_total']}")

In [ ]:
# ── Parse Logs & Compute Throughput from CSV ──────────────────────────────────

def parse_switching_times(log_path: str) -> list[float]:
    """Extract switching/recreation times from a log file."""
    times = []
    with open(log_path) as f:
        for line in f:
            m = re.search(r"(?:switch completed|recreated) in (\d+\.?\d*)s", line)
            if m:
                times.append(float(m.group(1)))
    return times

def find_trace_csv(approach: str) -> str | None:
    """Find the latest CSV trace file for an approach."""
    patterns = [
        f"{TRACE_DIR}/spottolerance_offline_{approach}_{_model_key_underscore}_scenario_{SCENARIO}_*.csv",
        f"{TRACE_DIR}/spottolerance_offline_{approach}_scenario_{SCENARIO}_*.csv",
    ]
    for pat in patterns:
        matches = sorted(glob.glob(pat))
        if matches:
            return matches[-1]
    return None

def calculate_throughput_from_csv(csv_path: str) -> float:
    """Calculate throughput from CSV: completed requests within BENCHMARK_DURATION_MIN."""
    df = pd.read_csv(csv_path)
    t0 = df["ArrivalTime"].min()
    cutoff = t0 + BENCHMARK_DURATION_MIN * 60
    completed = df[df["CompletionTime"] <= cutoff]
    duration_sec = BENCHMARK_DURATION_MIN * 60
    return len(completed) / duration_sec

switching_times = {}
throughputs = {}
for approach in APPROACHES:
    # Switching times from logs
    log_path = os.path.join(LOG_DIR, f"{approach}.log")
    if os.path.exists(log_path):
        st = parse_switching_times(log_path)
        if st:
            switching_times[approach] = st

    # Throughput from CSV trace
    csv_path = find_trace_csv(approach)
    if csv_path:
        throughputs[approach] = calculate_throughput_from_csv(csv_path)

print("Switching/recreation times (seconds):")
for approach, times in switching_times.items():
    print(f"  {approach}: {times}")
print(f"\nThroughput (req/s) — based on {BENCHMARK_DURATION_MIN} min cutoff:")
for approach, tput in throughputs.items():
    print(f"  {approach}: {tput:.2f}")

In [ ]:
# ── Compute Active Periods & Outgoing Nodes ───────────────────────────────────

# Filter events within benchmark duration
events = [e for e in events_raw if e["time_min"] < BENCHMARK_DURATION_MIN]

# Group events by time_min (compound events → one switch operation)
events_sorted = sorted(events, key=lambda e: e["time_min"])
compound_events = []
for t, grp in groupby(events_sorted, key=lambda e: e["time_min"]):
    compound_events.append({"time_min": t, "sub_events": list(grp)})

print(f"Compound events: {len(compound_events)}")
for ce in compound_events:
    types = [e["type"] for e in ce["sub_events"]]
    print(f"  t={ce['time_min']}min: {types}")

# ── Build timeline per instance type ──────────────────────────────────────────
# Each instance type gets a list of (time_min, spot_active, od_active) segments.

def count_event_spot_nodes(sub_event: dict) -> dict[str, int]:
    """Count spot nodes per instance type in a sub-event."""
    counts = defaultdict(int)
    for inst in sub_event["instances"]:
        group = node_to_group(inst)
        if is_spot_group(group):
            itype = group_to_instance_type(group)
            counts[itype] += 1
    return dict(counts)

# Track state and build segments
state = {itype: {"spot": s["spot_active"], "od": s["od_active"]}
         for itype, s in initial_state.items()}

segments = {itype: [] for itype in instance_types}  # list of (t_start, t_end, spot_active, od_active)
last_time = 0

for ce in compound_events:
    t = ce["time_min"]
    # Record segment from last_time to t with current state
    if t > last_time:
        for itype in instance_types:
            segments[itype].append((last_time, t, state[itype]["spot"], state[itype]["od"]))

    # Apply events
    for sub_event in ce["sub_events"]:
        spot_counts = count_event_spot_nodes(sub_event)
        for itype, n in spot_counts.items():
            if sub_event["type"] == "interruption":
                state[itype]["spot"] -= n
                state[itype]["od"] += n
            else:  # restore
                state[itype]["spot"] += n
                state[itype]["od"] -= n
    last_time = t

# Final segment to benchmark end
for itype in instance_types:
    segments[itype].append((last_time, BENCHMARK_DURATION_MIN, state[itype]["spot"], state[itype]["od"]))

# ── Compute outgoing nodes per compound event (for overlap) ───────────────────
# Outgoing = nodes being replaced. They continue running during overlap.
# interruption: outgoing = spot nodes (they linger while on-demand spins up)
# restore: outgoing = on-demand nodes (they linger while spot takes over)

overlap_state = {itype: {"spot": s["spot_active"], "od": s["od_active"]}
                 for itype, s in initial_state.items()}

compound_outgoing = []  # list of dicts: {itype: {"count": N, "is_spot": bool}}
for ce in compound_events:
    outgoing = {}
    for sub_event in ce["sub_events"]:
        spot_counts = count_event_spot_nodes(sub_event)
        for itype, n in spot_counts.items():
            if sub_event["type"] == "interruption":
                outgoing[itype] = {"count": n, "is_spot": True}
                overlap_state[itype]["spot"] -= n
                overlap_state[itype]["od"] += n
            else:  # restore
                outgoing[itype] = {"count": n, "is_spot": False}
                overlap_state[itype]["spot"] += n
                overlap_state[itype]["od"] -= n
    compound_outgoing.append(outgoing)

# ── Print timeline ────────────────────────────────────────────────────────────
print("\nTimeline per instance type:")
for itype in sorted(instance_types):
    print(f"\n  {itype}:")
    for t_start, t_end, s, od in segments[itype]:
        print(f"    [{t_start:2d}-{t_end:2d}] min  spot={s}, od={od}")

In [ ]:
# ── Price Lookup ───────────────────────────────────────────────────────────────

def get_spot_prices_by_az(instance_type: str) -> dict[str, float]:
    """Get average spot price per AZ for an instance type."""
    az_prices = defaultdict(list)
    for entry in prices_data["spot"]:
        if entry["Instance"] == instance_type:
            az_prices[entry["AZ"]].append(float(entry["Price"]))
    return {az: sum(ps) / len(ps) for az, ps in az_prices.items()}

def get_ondemand_price(instance_type: str) -> float:
    """Get on-demand price for an instance type."""
    for entry in prices_data["ondemand"]:
        if entry["Instance"] == instance_type:
            return float(entry["PricePerHour_USD"])
    raise ValueError(f"On-demand price not found for {instance_type}")

def get_spot_price(instance_type: str, mode: str = "avg") -> float:
    """Get spot price based on AZ selection and mode (avg/min/max)."""
    az_prices = get_spot_prices_by_az(instance_type)
    if not az_prices:
        raise ValueError(f"No spot price data for {instance_type}")

    # If AZ is specified for this instance type, use it directly
    if instance_type in AZ_SELECTION and AZ_SELECTION[instance_type]:
        az = AZ_SELECTION[instance_type]
        if az in az_prices:
            return az_prices[az]
        raise ValueError(f"AZ {az} not found for {instance_type}")

    all_prices = list(az_prices.values())
    if mode == "avg":
        return sum(all_prices) / len(all_prices)
    elif mode == "min":
        return min(all_prices)
    elif mode == "max":
        return max(all_prices)
    raise ValueError(f"Unknown mode: {mode}")

# Show prices
print("Spot prices by AZ:")
for itype in sorted(instance_types):
    az_prices = get_spot_prices_by_az(itype)
    if az_prices:
        for az, price in sorted(az_prices.items()):
            print(f"  {itype} [{az}]: ${price:.4f}/hr")

print("\nOn-demand prices:")
for itype in sorted(instance_types):
    try:
        print(f"  {itype}: ${get_ondemand_price(itype):.4f}/hr")
    except ValueError:
        print(f"  {itype}: N/A")

In [ ]:
# ── Cost Calculation ───────────────────────────────────────────────────────────

def calculate_all_costs(mode: str = "avg") -> dict[str, float]:
    """Calculate cost for each approach under a given spot price mode."""

    # ── 1. ondemand_only: all initial pipeline nodes at on-demand price ───────
    # Count total initial pipeline nodes per instance type (regardless of spot/od)
    pipeline_node_counts = defaultdict(int)
    for itype in instance_types:
        s = initial_state[itype]
        pipeline_node_counts[itype] = s["spot_active"] + s["od_active"]

    ondemand_cost = 0.0
    for itype, count in pipeline_node_counts.items():
        if count > 0:
            ondemand_cost += count * get_ondemand_price(itype) * BENCHMARK_DURATION_MIN / 60

    # ── 2. Base mixed cost (common to all event-driven approaches) ────────────
    # Sum cost over all segments for each instance type
    base_cost = 0.0
    for itype in instance_types:
        spot_price = get_spot_price(itype, mode) if initial_state[itype]["spot_total"] > 0 else 0
        od_price = get_ondemand_price(itype) if initial_state[itype]["od_total"] > 0 else 0

        for t_start, t_end, spot_n, od_n in segments[itype]:
            duration_hr = (t_end - t_start) / 60
            base_cost += spot_n * spot_price * duration_hr
            base_cost += od_n * od_price * duration_hr

    # ── 3. Overlap cost per approach ──────────────────────────────────────────
    results = {"only_ondemand": ondemand_cost}

    for approach in APPROACHES:
        if approach == "only_ondemand":
            continue

        total_cost = base_cost

        if approach in OVERLAP_APPROACHES and approach in switching_times:
            for i, outgoing in enumerate(compound_outgoing):
                if i < len(switching_times[approach]):
                    switch_sec = switching_times[approach][i]
                    for itype, info in outgoing.items():
                        if info["is_spot"]:
                            price = get_spot_price(itype, mode)
                        else:
                            price = get_ondemand_price(itype)
                        total_cost += info["count"] * price * switch_sec / 3600

        results[approach] = total_cost

    return results

# ── Compute for all modes ─────────────────────────────────────────────────────
# If all AZs are specified, avg/min/max will be identical
all_az_specified = all(
    itype in AZ_SELECTION and AZ_SELECTION[itype]
    for itype in instance_types
    if initial_state[itype]["spot_total"] > 0
)

modes = ["avg"] if all_az_specified else ["avg", "min", "max"]
cost_results = {mode: calculate_all_costs(mode) for mode in modes}

for mode in modes:
    print(f"\n{'='*60}")
    print(f"  Spot price mode: {mode.upper()}")
    print(f"{'='*60}")
    for approach, cost in cost_results[mode].items():
        print(f"  {approach:30s}  ${cost:.4f}")

In [ ]:
# ── Summary Table ──────────────────────────────────────────────────────────────

rows = []
for mode in modes:
    for approach in APPROACHES:
        cost = cost_results[mode].get(approach)
        tput = throughputs.get(approach)
        row = {
            "Approach": approach,
            "Spot Price Mode": mode,
            "Cost ($)": f"{cost:.4f}" if cost is not None else "N/A",
        }
        if tput is not None:
            row["Throughput (req/s)"] = f"{tput:.2f}"
            row["Throughput/Cost (req/s/$)"] = f"{tput / cost:.4f}" if cost else "N/A"
        else:
            row["Throughput (req/s)"] = "N/A"
            row["Throughput/Cost (req/s/$)"] = "N/A"
        rows.append(row)

df = pd.DataFrame(rows)
df

In [ ]:
# ── Cost Comparison Bar Chart ─────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Display config
DISPLAY_NAMES = {
    "only_ondemand": "On-demand\nOnly",
    "no_handle": "No\nHandle",
    "request_migration": "Request\nMigration",
    "concurrent_initialization": "Concurrent\nInitialization",
    "shuntserve": "ShuntServe",
}
PLOT_ORDER = ["only_ondemand", "no_handle", "request_migration",
              "concurrent_initialization", "shuntserve"]
COLORS = {
    "only_ondemand": "#808080",
    "no_handle": "#d62728",
    "request_migration": "#2ca02c",
    "concurrent_initialization": "#ff7f0e",
    "shuntserve": "#1f77b4",
}
HATCHES = {
    "only_ondemand": "",
    "no_handle": "/",
    "request_migration": "\\",
    "concurrent_initialization": "x",
    "shuntserve": ".",
}

FIGURES_DIR = "figures"

# Use the first available mode (if AZ specified → only "avg")
mode = modes[0]
costs_dict = cost_results[mode]

labels = [DISPLAY_NAMES[a] for a in PLOT_ORDER]
values = [costs_dict[a] for a in PLOT_ORDER]

fontsize = 35
fig, ax = plt.subplots(figsize=(8, 6))

for i, a in enumerate(PLOT_ORDER):
    ax.bar(i, costs_dict[a], width=0.8, color=COLORS[a], alpha=0.7,
           edgecolor="black", linewidth=1.5, hatch=HATCHES[a])

# Value labels
for i, v in enumerate(values):
    ax.text(i, v + max(values) * 0.02, f"{v:.2f}",
            ha="center", va="bottom", fontsize=fontsize, rotation=90)

ax.set_ylabel("Cost ($)", fontsize=fontsize)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([])
ax.set_xlabel(" ", fontsize=fontsize)
ax.set_ylim(0, max(values) * 1.45)
ax.tick_params(axis="y", labelsize=fontsize)
ax.grid(True, axis="y", alpha=0.3, linestyle="--")
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/cost_comparison_scenario_{SCENARIO}.pdf", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR}/cost_comparison_scenario_{SCENARIO}.pdf")

In [ ]:
# ── Legend (separate file for paper figures) ──────────────────────────────────
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 0.5))
ax.axis("off")

legend_elements = [
    mpatches.Patch(facecolor=COLORS[a], edgecolor="black", alpha=0.7,
                   hatch=HATCHES[a], label=DISPLAY_NAMES[a].replace("\n", " "))
    for a in PLOT_ORDER
]
ax.legend(handles=legend_elements, fontsize=20, loc="center", ncol=5,
          columnspacing=0.8, handlelength=1.5, edgecolor="black")

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/cost_legend_scenario_{SCENARIO}.pdf", dpi=300, bbox_inches="tight", transparent=True)
plt.show()
print(f"Saved: {FIGURES_DIR}/cost_legend_scenario_{SCENARIO}.pdf")